# RanSMAP Semantic Embedding Extraction (from Behavioural Features)

This notebook builds **384-dimensional semantic embeddings directly from the RanSMAP hardware
telemetry dataset** — the same source used in `01_behavioural_feature_extraction.ipynb`.

**Motivation:** The PE-file semantic dataset (`02_semantic_embedding_extraction.ipynb`) captures
static file structure. This notebook explores whether the *same hardware-level signals* used
for behavioural classification also carry a useful semantic representation when expressed
as natural language — allowing the Semantic Agent to be trained and compared on identical
data to the Behavioural Agent.

**Approach:** Each 10-second window from `combined_i3-gen12_datasetF.parquet` is serialised
into a short descriptive sentence of its non-zero hardware metrics, then encoded by
`sentence-transformers/all-MiniLM-L6-v2`.

**Input:** `combined_i3-gen12_datasetF.parquet` (output of notebook 01)  
**Output:** `ransmap_semantic_embeddings.parquet` — same rows + `semantic_text` + `embedding` (384-dim)

### Difference from notebook 02
| | Notebook 02 | This notebook |
|--|-------------|---------------|
| Source dataset | PE file metadata (Ransomware.csv) | RanSMAP hardware telemetry |
| Feature type | Static analysis (imports, entropy, sections) | Dynamic hardware traces (throughput, GPA, memory counts) |
| Label source | `legitimate` column (inverted) | `label` column (already correct) |
| Use case | Cross-dataset semantic agent | Same-dataset semantic vs behavioural comparison |

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_PARQUET  = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet"
OUTPUT_PARQUET = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/ransmap_semantic_embeddings.parquet"

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 256

# ── Text generation ──────────────────────────────────────────────────────────
# Metadata columns excluded from the sentence — not hardware signals
EXCLUDE_COLS = {"label", "category", "timestamp"}

# Human-readable aliases for the 19 behavioural features
PRETTY_MAP = {
    "avg_read_throughput":        "average disk read throughput",
    "avg_write_throughput":       "average disk write throughput",
    "read_LBA_variance":          "read LBA address variance",
    "write_LBA_variance":         "write LBA address variance",
    "write_entropy_mean":         "write operation entropy",
    "mem_read_entropy_mean":      "memory read size entropy",
    "mem_write_entropy_mean":     "memory write size entropy",
    "mem_readwrite_entropy_mean": "memory read-write size entropy",
    "mem_read_GPA_variance":      "memory read address variance",
    "mem_write_GPA_variance":     "memory write address variance",
    "mem_readwrite_GPA_variance": "memory read-write address variance",
    "mem_exec_GPA_variance":      "memory exec address variance",
    "mem_read_count":             "memory read operation count",
    "mem_write_count":            "memory write operation count",
    "mem_readwrite_count":        "memory read-write operation count",
    "mem_exec_count":             "memory exec operation count",
    "access_type_2_count":        "access type 2 count",
    "access_type_4_count":        "access type 4 count",
    "mem_4KiB_page_access_count": "4KiB memory page access count",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"Input   : {INPUT_PARQUET}")
print(f"Output  : {OUTPUT_PARQUET}")
print(f"Model   : {MODEL_NAME}")

## 2. Load the Behavioural Feature Dataset

In [ ]:
df = pd.read_parquet(INPUT_PARQUET)

print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"\nLabel distribution (0=benign, 1=ransomware):")
print(df["label"].value_counts())
print(f"\nClass distribution:")
print(df["category"].value_counts())
df.head(3)

## 3. Convert Each Window to a Natural-Language Sentence

Zero-valued features are skipped — a window where no disk writes occurred should not
say *"average disk write throughput 0"*, since that adds noise rather than signal.
The resulting sentence describes only what actually happened in that 10-second window.

Example output:  
`"average disk write throughput 85432.1; write operation entropy 0.9231; memory write address variance 1.2e+09; memory write operation count 342.0"`

In [ ]:
def row_to_text(row: pd.Series) -> str:
    parts: List[str] = []
    for col, val in row.items():
        if col in EXCLUDE_COLS or pd.isna(val):
            continue
        if isinstance(val, (int, float)) and val == 0:
            continue
        label = PRETTY_MAP.get(col, col.replace("_", " "))
        parts.append(f"{label} {round(float(val), 4)}")
    return "; ".join(parts) if parts else "no hardware activity"


print("Generating semantic text representations...")
df["semantic_text"] = df.apply(row_to_text, axis=1)

print(f"Done. Sample sentences:")
for i, (idx, row) in enumerate(df[["category", "label", "semantic_text"]].iloc[:4].iterrows()):
    print(f"  [{row['category']} | label={row['label']}]")
    print(f"    {row['semantic_text'][:130]}")
    print()

# Check for empty sentences (windows with all-zero features)
empty = (df["semantic_text"] == "no hardware activity").sum()
print(f"Windows with no activity: {empty:,} / {len(df):,}")

## 4. Load the Sentence Transformer Model

In [ ]:
print(f"Loading model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device=str(device))

print(f"Running on   : {device}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 5. Generate Embeddings

In [ ]:
texts = df["semantic_text"].tolist()
print(f"Encoding {len(texts):,} samples (batch_size={BATCH_SIZE})...")

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)  # [N, 384]

print(f"\nEmbedding matrix shape : {embeddings.shape}")
print(f"dtype                  : {embeddings.dtype}")
print(f"Mean L2 norm           : {np.linalg.norm(embeddings, axis=1).mean():.4f}")

## 6. Attach Embeddings to the DataFrame

In [ ]:
df["embedding"] = embeddings.tolist()

print(f"Final DataFrame shape  : {df.shape}")
print(f"Embedding vector length: {len(df['embedding'].iloc[0])}")
df[["category", "label", "semantic_text", "embedding"]].head(2)

## 7. Sanity Check — Class Separation in Embedding Space

Compares within-class and cross-class cosine similarity.
Also checks per-class sentence length to confirm ransomware windows
generate richer descriptions (more non-zero features) than benign ones.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

benign_emb = embeddings[df["label"].values == 0][:100]
ransom_emb = embeddings[df["label"].values == 1][:100]

if len(benign_emb) >= 2 and len(ransom_emb) >= 2:
    sim_within_benign = cosine_similarity(benign_emb).mean()
    sim_within_ransom = cosine_similarity(ransom_emb).mean()
    sim_cross         = cosine_similarity(benign_emb, ransom_emb).mean()

    print(f"Within-benign similarity    : {sim_within_benign:.4f}")
    print(f"Within-ransomware similarity: {sim_within_ransom:.4f}")
    print(f"Cross-class similarity      : {sim_cross:.4f}")
    print("\n(Lower cross-class = better separation in embedding space)")

# Sentence length as a proxy for feature richness per class
df["text_len"] = df["semantic_text"].str.len()
print(f"\nMean sentence length per class:")
print(df.groupby("label")["text_len"].mean().rename({0: "benign", 1: "ransomware"}))
df.drop(columns=["text_len"], inplace=True)

## 8. Save to Parquet

In [ ]:
Path(OUTPUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Saved {len(df):,} rows → {OUTPUT_PARQUET}")

# Reload and verify
verify = pd.read_parquet(OUTPUT_PARQUET)
assert verify.shape == df.shape, f"[WARN] Shape mismatch: {verify.shape} != {df.shape}"
assert len(verify["embedding"].iloc[0]) == 384, "[WARN] Unexpected embedding dimension!"
print(f"Reload check   : {verify.shape} — OK")
print(f"Columns saved  : {verify.columns.tolist()}")